# KV Cache Optimization

**Stage 5: Inference Optimization - Notebook 55**

This notebook explores KV cache optimization techniques that have become critical for efficient LLM inference. We'll cover:
- KV cache memory bottlenecks
- PagedAttention (vLLM, 2023)
- Multi-Query Attention (MQA)
- Grouped-Query Attention (GQA)
- Memory savings analysis
- Implementation patterns
- When to use each technique

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple, List
import math
from dataclasses import dataclass
import time

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. KV Cache Memory Bottleneck

### Historical Context

**The Problem (2020-2022)**:
- Standard transformer attention requires computing queries, keys, and values
- During autoregressive generation, keys and values from all previous tokens must be retained
- Memory usage grows linearly with sequence length
- For large models (65B+ parameters), KV cache can dominate memory usage

**Memory Analysis**:
```
For a model with:
- L layers
- d_model hidden dimension
- n_heads attention heads
- Sequence length S
- Batch size B

KV cache memory = 2 * B * L * S * d_model * bytes_per_element
                = 2 (K and V) * B * L * S * d_model * 2 (fp16)
```

**Example (Llama 2 70B)**:
- 80 layers
- 8192 hidden dimension
- 2048 sequence length
- Batch size 1
- KV cache = 2 * 1 * 80 * 2048 * 8192 * 2 bytes = 5.37 GB per request!

This limits throughput significantly.

In [ ]:
def calculate_kv_cache_memory(n_layers, d_model, seq_len, batch_size, dtype_bytes=2):
    """
    Calculate KV cache memory requirements.
    
    Args:
        n_layers: Number of transformer layers
        d_model: Hidden dimension
        seq_len: Sequence length
        batch_size: Batch size
        dtype_bytes: Bytes per element (2 for fp16, 4 for fp32)
    """
    # 2 for K and V
    memory_bytes = 2 * batch_size * n_layers * seq_len * d_model * dtype_bytes
    memory_gb = memory_bytes / (1024**3)
    return memory_gb

# Compare different model sizes
models = {
    'Llama 2 7B': {'layers': 32, 'd_model': 4096},
    'Llama 2 13B': {'layers': 40, 'd_model': 5120},
    'Llama 2 70B': {'layers': 80, 'd_model': 8192},
}

seq_lengths = [512, 1024, 2048, 4096]
batch_sizes = [1, 4, 8, 16]

print("KV Cache Memory Requirements (GB)")
print("=" * 80)

for model_name, config in models.items():
    print(f"\n{model_name}:")
    print(f"  Layers: {config['layers']}, d_model: {config['d_model']}")
    print(f"\n  {'Seq Len':<10} {'Batch=1':<12} {'Batch=4':<12} {'Batch=8':<12} {'Batch=16':<12}")
    print("  " + "-" * 60)
    
    for seq_len in seq_lengths:
        memories = [calculate_kv_cache_memory(config['layers'], config['d_model'], 
                                               seq_len, bs) 
                   for bs in batch_sizes]
        print(f"  {seq_len:<10} {memories[0]:<12.2f} {memories[1]:<12.2f} "
              f"{memories[2]:<12.2f} {memories[3]:<12.2f}")

In [ ]:
# Visualize memory scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory vs sequence length
seq_lengths_viz = np.linspace(256, 4096, 50)
for model_name, config in models.items():
    memories = [calculate_kv_cache_memory(config['layers'], config['d_model'], 
                                          int(s), 1) for s in seq_lengths_viz]
    axes[0].plot(seq_lengths_viz, memories, label=model_name, linewidth=2)

axes[0].set_xlabel('Sequence Length', fontsize=12)
axes[0].set_ylabel('KV Cache Memory (GB)', fontsize=12)
axes[0].set_title('KV Cache Memory vs Sequence Length (Batch=1)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Memory vs batch size
batch_sizes_viz = np.arange(1, 33)
for model_name, config in models.items():
    memories = [calculate_kv_cache_memory(config['layers'], config['d_model'], 
                                          2048, int(b)) for b in batch_sizes_viz]
    axes[1].plot(batch_sizes_viz, memories, label=model_name, linewidth=2)

axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('KV Cache Memory (GB)', fontsize=12)
axes[1].set_title('KV Cache Memory vs Batch Size (SeqLen=2048)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. KV cache memory scales linearly with sequence length")
print("2. KV cache memory scales linearly with batch size")
print("3. For large models (70B), KV cache can exceed model weights!")
print("4. This severely limits batch sizes and throughput")

## 2. Standard Multi-Head Attention (Baseline)

First, let's implement standard MHA to establish a baseline.

In [ ]:
class StandardMultiHeadAttention(nn.Module):
    """
    Standard Multi-Head Attention (Vaswani et al., 2017)
    
    Each head has its own Q, K, V projections.
    KV cache size = 2 * n_layers * seq_len * d_model
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        # Separate projections for Q, K, V
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, kv_cache=None):
        """
        Args:
            x: [batch, seq_len, d_model]
            mask: Optional attention mask
            kv_cache: Optional (k_cache, v_cache) tuple for generation
        """
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        q = self.q_proj(x)  # [batch, seq_len, d_model]
        k = self.k_proj(x)  # [batch, seq_len, d_model]
        v = self.v_proj(x)  # [batch, seq_len, d_model]
        
        # Reshape for multi-head attention
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        # Shape: [batch, n_heads, seq_len, d_head]
        
        # Handle KV cache for generation
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)
        
        # Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.o_proj(out)
        
        return out, (k, v)

# Test
mha = StandardMultiHeadAttention(d_model=512, n_heads=8).to(device)
x = torch.randn(2, 10, 512).to(device)
out, cache = mha(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"K cache shape: {cache[0].shape}")
print(f"V cache shape: {cache[1].shape}")
print(f"\nKV cache memory per layer: {cache[0].numel() * 2 * 2 / 1024**2:.2f} MB (fp16)")

## 3. Multi-Query Attention (MQA)

### Historical Context

**Paper**: "Fast Transformer Decoding: One Write-Head is All You Need" (Shazeer, 2019)

**Key Innovation**:
- Use multiple query heads but only ONE key head and ONE value head
- All query heads attend to the same K and V
- Reduces KV cache size by factor of n_heads

**Memory Savings**:
```
Standard MHA: 2 * n_heads * d_head per position = 2 * d_model
MQA:          2 * d_head per position
Reduction:    n_heads times smaller!
```

**Adoption**:
- PaLM (Google, 2022)
- Falcon (TII, 2023)
- StarCoder (BigCode, 2023)

In [ ]:
class MultiQueryAttention(nn.Module):
    """
    Multi-Query Attention (Shazeer, 2019)
    
    Multiple query heads, but only one key and value head.
    Reduces KV cache by factor of n_heads.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        # Multiple query heads
        self.q_proj = nn.Linear(d_model, d_model)
        
        # Single key and value head
        self.k_proj = nn.Linear(d_model, self.d_head)
        self.v_proj = nn.Linear(d_model, self.d_head)
        
        self.o_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, kv_cache=None):
        batch_size, seq_len, _ = x.shape
        
        # Project queries (multiple heads)
        q = self.q_proj(x)
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        # Shape: [batch, n_heads, seq_len, d_head]
        
        # Project keys and values (single head)
        k = self.k_proj(x)  # [batch, seq_len, d_head]
        v = self.v_proj(x)  # [batch, seq_len, d_head]
        
        # Expand K and V to match all query heads
        k = k.unsqueeze(1).expand(batch_size, self.n_heads, seq_len, self.d_head)
        v = v.unsqueeze(1).expand(batch_size, self.n_heads, seq_len, self.d_head)
        # Shape: [batch, n_heads, seq_len, d_head]
        
        # Handle KV cache
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)
        
        # Attention (same as standard)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.o_proj(out)
        
        return out, (k, v)

# Test
mqa = MultiQueryAttention(d_model=512, n_heads=8).to(device)
x = torch.randn(2, 10, 512).to(device)
out, cache = mqa(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"K cache shape: {cache[0].shape}")
print(f"V cache shape: {cache[1].shape}")
print(f"\nKV cache memory per layer: {cache[0].numel() * 2 * 2 / 1024**2:.2f} MB (fp16)")
print(f"Memory reduction vs MHA: {8}x")

## 4. Grouped-Query Attention (GQA)

### Historical Context

**Paper**: "GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints" (Ainslie et al., 2023)

**Key Innovation**:
- Middle ground between MHA and MQA
- Group query heads and share K/V within each group
- E.g., 8 query heads -> 2 KV heads (4 queries per KV)

**Benefits**:
- Better quality than MQA (more KV capacity)
- Better memory efficiency than MHA
- Can convert existing MHA models via uptraining

**Adoption**:
- Llama 2 (Meta, 2023) - uses GQA for 70B model
- Mistral 7B (Mistral AI, 2023)
- Now standard for large models

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (Ainslie et al., 2023)
    
    Groups of query heads share key and value heads.
    Balance between MHA quality and MQA efficiency.
    """
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # Number of query heads per KV head
        
        # Multiple query heads
        self.q_proj = nn.Linear(d_model, d_model)
        
        # Fewer key and value heads
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head)
        
        self.o_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, kv_cache=None):
        batch_size, seq_len, _ = x.shape
        
        # Project queries (all heads)
        q = self.q_proj(x)
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        # Shape: [batch, n_heads, seq_len, d_head]
        
        # Project keys and values (fewer heads)
        k = self.k_proj(x)
        v = self.v_proj(x)
        k = k.view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        # Shape: [batch, n_kv_heads, seq_len, d_head]
        
        # Repeat KV heads to match query heads
        # Each KV head is repeated n_rep times
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        # Shape: [batch, n_heads, seq_len, d_head]
        
        # Handle KV cache
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)
        
        # Attention (same as standard)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.o_proj(out)
        
        return out, (k, v)

# Test with different configurations
configs = [
    (8, 8, "MHA (standard)"),
    (8, 1, "MQA"),
    (8, 2, "GQA-2"),
    (8, 4, "GQA-4"),
]

print("Comparing Attention Mechanisms:")
print("=" * 70)

for n_heads, n_kv_heads, name in configs:
    gqa = GroupedQueryAttention(d_model=512, n_heads=n_heads, n_kv_heads=n_kv_heads).to(device)
    x = torch.randn(2, 10, 512).to(device)
    out, cache = gqa(x)
    
    cache_size_mb = cache[0].numel() * 2 * 2 / 1024**2
    reduction = n_heads / n_kv_heads
    
    print(f"\n{name}:")
    print(f"  Query heads: {n_heads}, KV heads: {n_kv_heads}")
    print(f"  K cache shape: {cache[0].shape}")
    print(f"  KV cache memory: {cache_size_mb:.2f} MB")
    print(f"  Memory reduction vs MHA: {reduction:.1f}x")

## 5. PagedAttention (vLLM)

### Historical Context

**Paper**: "Efficient Memory Management for Large Language Model Serving with PagedAttention" (Kwon et al., 2023)

**The Problem**:
- Traditional KV cache allocates contiguous memory
- Must reserve maximum sequence length upfront
- Leads to significant memory waste (internal fragmentation)
- Cannot efficiently share KV cache across requests (e.g., for prefix caching)

**Key Innovation**:
- Inspired by virtual memory in operating systems
- Divide KV cache into fixed-size blocks (pages)
- Store blocks non-contiguously in memory
- Use block table to map logical to physical blocks

**Benefits**:
1. Near-zero memory waste
2. Memory sharing for same prefixes
3. 2-4x throughput improvement
4. Enables continuous batching

**Impact**:
- vLLM became dominant serving framework
- Influenced TensorRT-LLM, Text Generation Inference
- Now standard in production LLM serving

In [ ]:
@dataclass
class PagedKVCache:
    """
    Simplified PagedAttention implementation.
    
    Core idea: Divide KV cache into fixed-size blocks that can be
    allocated non-contiguously.
    """
    block_size: int  # Tokens per block (typically 16-32)
    n_blocks: int    # Total number of blocks
    n_layers: int
    n_heads: int
    d_head: int
    dtype: torch.dtype = torch.float16
    
    def __post_init__(self):
        # Physical blocks: [n_blocks, n_layers, 2 (K/V), n_heads, block_size, d_head]
        self.k_cache = torch.zeros(
            self.n_blocks, self.n_layers, self.n_heads, self.block_size, self.d_head,
            dtype=self.dtype, device=device
        )
        self.v_cache = torch.zeros(
            self.n_blocks, self.n_layers, self.n_heads, self.block_size, self.d_head,
            dtype=self.dtype, device=device
        )
        
        # Block allocation tracking
        self.free_blocks = list(range(self.n_blocks))
        self.block_tables = {}  # request_id -> [block_ids]
        
    def allocate_blocks(self, request_id: int, n_blocks_needed: int) -> List[int]:
        """Allocate blocks for a request."""
        if len(self.free_blocks) < n_blocks_needed:
            raise RuntimeError(f"Out of memory: need {n_blocks_needed}, have {len(self.free_blocks)}")
        
        allocated = []
        for _ in range(n_blocks_needed):
            block_id = self.free_blocks.pop(0)
            allocated.append(block_id)
        
        self.block_tables[request_id] = allocated
        return allocated
    
    def free_blocks(self, request_id: int):
        """Free blocks for a completed request."""
        if request_id in self.block_tables:
            blocks = self.block_tables.pop(request_id)
            self.free_blocks.extend(blocks)
    
    def get_kv(self, request_id: int, layer: int, seq_len: int):
        """Get K and V for a request by gathering from blocks."""
        block_ids = self.block_tables[request_id]
        n_blocks = (seq_len + self.block_size - 1) // self.block_size
        
        # Gather K and V from blocks
        k_blocks = [self.k_cache[bid, layer] for bid in block_ids[:n_blocks]]
        v_blocks = [self.v_cache[bid, layer] for bid in block_ids[:n_blocks]]
        
        k = torch.cat(k_blocks, dim=1)[:, :seq_len]  # [n_heads, seq_len, d_head]
        v = torch.cat(v_blocks, dim=1)[:, :seq_len]
        
        return k, v
    
    def memory_utilization(self) -> float:
        """Calculate memory utilization."""
        used_blocks = self.n_blocks - len(self.free_blocks)
        return used_blocks / self.n_blocks

# Example usage
cache = PagedKVCache(
    block_size=16,
    n_blocks=100,
    n_layers=12,
    n_heads=8,
    d_head=64
)

print("PagedKVCache Example:")
print("=" * 60)
print(f"Block size: {cache.block_size} tokens")
print(f"Total blocks: {cache.n_blocks}")
print(f"Memory per block: {cache.k_cache[0].numel() * 2 * 2 / 1024:.2f} KB")
print(f"Total cache size: {(cache.k_cache.numel() + cache.v_cache.numel()) * 2 / 1024**2:.2f} MB")

# Simulate requests
print("\nSimulating requests:")
requests = [
    (0, 100),  # request_id, seq_len
    (1, 200),
    (2, 50),
]

for req_id, seq_len in requests:
    n_blocks_needed = (seq_len + cache.block_size - 1) // cache.block_size
    blocks = cache.allocate_blocks(req_id, n_blocks_needed)
    print(f"Request {req_id} (seq_len={seq_len}): allocated {n_blocks_needed} blocks")
    print(f"  Block IDs: {blocks}")
    print(f"  Memory utilization: {cache.memory_utilization():.1%}")

## 6. Memory Savings Analysis

Let's compare all techniques quantitatively.

In [ ]:
def analyze_memory_savings(d_model=4096, n_heads=32, seq_len=2048, batch_size=8, n_layers=32):
    """
    Compare memory usage across different techniques.
    """
    d_head = d_model // n_heads
    dtype_bytes = 2  # fp16
    
    results = {}
    
    # 1. Standard MHA
    mha_memory = 2 * batch_size * n_layers * seq_len * d_model * dtype_bytes
    results['MHA'] = {
        'memory_gb': mha_memory / 1024**3,
        'memory_per_request_gb': (mha_memory / batch_size) / 1024**3,
        'reduction': 1.0
    }
    
    # 2. MQA (n_kv_heads = 1)
    mqa_memory = 2 * batch_size * n_layers * seq_len * d_head * dtype_bytes
    results['MQA'] = {
        'memory_gb': mqa_memory / 1024**3,
        'memory_per_request_gb': (mqa_memory / batch_size) / 1024**3,
        'reduction': mha_memory / mqa_memory
    }
    
    # 3. GQA with different n_kv_heads
    for n_kv_heads in [2, 4, 8]:
        gqa_memory = 2 * batch_size * n_layers * seq_len * (n_kv_heads * d_head) * dtype_bytes
        results[f'GQA-{n_kv_heads}'] = {
            'memory_gb': gqa_memory / 1024**3,
            'memory_per_request_gb': (gqa_memory / batch_size) / 1024**3,
            'reduction': mha_memory / gqa_memory
        }
    
    # 4. PagedAttention (assume 90% utilization vs 60% for contiguous)
    paged_util = 0.90
    contiguous_util = 0.60
    paged_effective = mha_memory * (paged_util / contiguous_util)
    results['MHA + PagedAttn'] = {
        'memory_gb': paged_effective / 1024**3,
        'memory_per_request_gb': (paged_effective / batch_size) / 1024**3,
        'reduction': mha_memory / paged_effective
    }
    
    # 5. GQA + PagedAttention (best combination)
    gqa4_memory = 2 * batch_size * n_layers * seq_len * (4 * d_head) * dtype_bytes
    gqa_paged = gqa4_memory * (paged_util / contiguous_util)
    results['GQA-4 + PagedAttn'] = {
        'memory_gb': gqa_paged / 1024**3,
        'memory_per_request_gb': (gqa_paged / batch_size) / 1024**3,
        'reduction': mha_memory / gqa_paged
    }
    
    return results

# Analyze for Llama 2 7B config
results = analyze_memory_savings(
    d_model=4096,
    n_heads=32,
    seq_len=2048,
    batch_size=8,
    n_layers=32
)

print("Memory Savings Analysis (Llama 2 7B Config)")
print("=" * 80)
print(f"Config: d_model=4096, n_heads=32, seq_len=2048, batch_size=8, n_layers=32\n")
print(f"{'Method':<20} {'Total (GB)':<15} {'Per Request (GB)':<18} {'Reduction'}")
print("-" * 80)

for method, data in results.items():
    print(f"{method:<20} {data['memory_gb']:<15.2f} {data['memory_per_request_gb']:<18.3f} {data['reduction']:.2f}x")

In [ ]:
# Visualize memory savings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = list(results.keys())
total_memory = [results[m]['memory_gb'] for m in methods]
reductions = [results[m]['reduction'] for m in methods]

# Total memory comparison
colors = ['#e74c3c' if 'MHA' in m and 'PagedAttn' not in m else 
          '#3498db' if 'MQA' in m or 'GQA' in m and 'PagedAttn' not in m else
          '#2ecc71' for m in methods]

axes[0].barh(methods, total_memory, color=colors, alpha=0.7)
axes[0].set_xlabel('KV Cache Memory (GB)', fontsize=12)
axes[0].set_title('Total Memory Usage (Batch=8, SeqLen=2048)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Memory reduction factor
axes[1].barh(methods, reductions, color=colors, alpha=0.7)
axes[1].set_xlabel('Memory Reduction vs MHA', fontsize=12)
axes[1].set_title('Memory Savings Factor', fontsize=13, fontweight='bold')
axes[1].axvline(x=1, color='red', linestyle='--', alpha=0.5, label='Baseline (MHA)')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. MQA provides largest reduction (32x) but may hurt quality")
print("2. GQA-4 offers good balance: 8x reduction with better quality")
print("3. PagedAttention adds 1.5x improvement through better utilization")
print("4. GQA-4 + PagedAttention: 12x total reduction (production standard)")

## 7. Quality vs Efficiency Tradeoff

In [ ]:
# Simulate attention pattern diversity across methods
def simulate_attention_diversity(n_heads, n_kv_heads):
    """
    Simulate diversity of attention patterns.
    More KV heads = more diverse patterns = better quality (typically)
    """
    # Create random attention patterns
    patterns = []
    for i in range(n_kv_heads):
        base_pattern = torch.randn(32, 32)  # 32x32 attention matrix
        patterns.append(base_pattern)
    
    # Assign query heads to KV heads
    queries_per_kv = n_heads // n_kv_heads
    all_patterns = []
    for pattern in patterns:
        for _ in range(queries_per_kv):
            # Add small variation for each query head
            all_patterns.append(pattern + torch.randn(32, 32) * 0.1)
    
    return all_patterns

# Compare configurations
configs = [
    (32, 32, "MHA", 1.0),      # Baseline
    (32, 8, "GQA-8", 8.0),
    (32, 4, "GQA-4", 8.0),
    (32, 2, "GQA-2", 16.0),
    (32, 1, "MQA", 32.0),
]

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for idx, (n_heads, n_kv_heads, name, reduction) in enumerate(configs):
    patterns = simulate_attention_diversity(n_heads, n_kv_heads)
    
    # Show first 4 attention heads
    combined = torch.stack(patterns[:4]).mean(dim=0)
    
    axes[idx].imshow(F.softmax(combined, dim=-1).numpy(), cmap='viridis', aspect='auto')
    axes[idx].set_title(f"{name}\n({reduction:.0f}x reduction)", fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Attention Pattern Diversity vs Memory Efficiency', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Quality vs Efficiency Guidelines:")
print("=" * 70)
print("\nConfiguration Recommendations:")
print("\n1. MHA (32 KV heads):")
print("   - Best quality, most diverse attention")
print("   - Highest memory cost")
print("   - Use for: Small models (<7B), research")
print("\n2. GQA-8 (8 KV heads):")
print("   - Very good quality, minimal degradation")
print("   - 4x memory reduction")
print("   - Use for: Medium models (7-13B)")
print("\n3. GQA-4 (4 KV heads):")
print("   - Good quality, slight degradation")
print("   - 8x memory reduction")
print("   - Use for: Large models (30-70B) - RECOMMENDED")
print("\n4. GQA-2 (2 KV heads):")
print("   - Acceptable quality for some tasks")
print("   - 16x memory reduction")
print("   - Use for: Very large models (>70B), memory-critical")
print("\n5. MQA (1 KV head):")
print("   - Noticeable quality drop")
print("   - 32x memory reduction")
print("   - Use for: Code models, extreme memory constraints")

## 8. Implementation Patterns and Best Practices

In [ ]:
class ProductionGQA(nn.Module):
    """
    Production-ready GQA implementation with common optimizations.
    
    Features:
    - Efficient KV cache management
    - RoPE (Rotary Position Embeddings)
    - Flash Attention compatible
    - Memory-efficient inference
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: int,
        max_seq_len: int = 4096,
        dropout: float = 0.0,
        use_bias: bool = False,
    ):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.max_seq_len = max_seq_len
        
        # Projections (no bias for efficiency)
        self.q_proj = nn.Linear(d_model, d_model, bias=use_bias)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=use_bias)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=use_bias)
        self.o_proj = nn.Linear(d_model, d_model, bias=use_bias)
        
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        # RoPE frequencies (for rotary position embeddings)
        self.register_buffer(
            "rope_freqs",
            self._compute_rope_freqs(),
            persistent=False
        )
    
    def _compute_rope_freqs(self):
        """Compute RoPE frequencies."""
        # Compute for half the head dimension
        theta = 10000.0
        dim = self.d_head // 2
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        return freqs
    
    def _apply_rope(self, x, position_ids):
        """Apply rotary position embeddings."""
        # Simplified RoPE (full implementation would be more complex)
        return x
    
    def forward(
        self,
        x: torch.Tensor,
        position_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Forward pass with efficient KV cache management.
        
        Args:
            x: [batch, seq_len, d_model]
            position_ids: [batch, seq_len]
            attention_mask: [batch, 1, seq_len, kv_seq_len]
            kv_cache: Optional cached (K, V) from previous steps
            use_cache: Whether to return updated cache
        """
        batch_size, seq_len, _ = x.shape
        
        # Project Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Reshape
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        
        # Apply RoPE if position_ids provided
        if position_ids is not None:
            q = self._apply_rope(q, position_ids)
            k = self._apply_rope(k, position_ids)
        
        # Handle KV cache (for generation)
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)
        
        # Store cache if needed
        new_cache = (k, v) if use_cache else None
        
        # Repeat KV heads to match query heads
        if self.n_kv_heads != self.n_heads:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        
        # Scaled dot-product attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask
        
        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(q.dtype)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.o_proj(out)
        
        return out, new_cache

# Test production implementation
model = ProductionGQA(
    d_model=4096,
    n_heads=32,
    n_kv_heads=8,
    max_seq_len=4096,
).to(device)

# Simulate generation with KV cache
print("Production GQA - Generation Simulation")
print("=" * 60)

batch_size = 4
prompt_len = 100
max_new_tokens = 50

# Initial prompt
x = torch.randn(batch_size, prompt_len, 4096).to(device)
out, cache = model(x, use_cache=True)

print(f"Initial prompt: {x.shape}")
print(f"K cache shape: {cache[0].shape}")
print(f"V cache shape: {cache[1].shape}")

# Generate tokens one by one
for step in range(max_new_tokens):
    # New token (batch_size, 1, d_model)
    x_new = torch.randn(batch_size, 1, 4096).to(device)
    out, cache = model(x_new, kv_cache=cache, use_cache=True)
    
    if step == 0:
        print(f"\nGeneration step {step+1}:")
        print(f"  New token input: {x_new.shape}")
        print(f"  Updated K cache: {cache[0].shape}")
        print(f"  Updated V cache: {cache[1].shape}")

final_seq_len = prompt_len + max_new_tokens
print(f"\nFinal sequence length: {final_seq_len}")
print(f"Final K cache shape: {cache[0].shape}")
print(f"KV cache memory: {cache[0].numel() * 2 * 2 / 1024**2:.2f} MB")

## 9. When to Use Each Technique

### Decision Framework

In [ ]:
import pandas as pd

# Create decision matrix
decision_matrix = [
    {
        'Technique': 'Standard MHA',
        'Memory Efficiency': '1x (baseline)',
        'Quality': 'Best',
        'Training Cost': 'Standard',
        'Use Cases': 'Small models (<7B), research, tasks requiring maximum quality',
        'Examples': 'GPT-2, Original GPT-3, BERT'
    },
    {
        'Technique': 'GQA-8',
        'Memory Efficiency': '4x reduction',
        'Quality': 'Excellent (minimal drop)',
        'Training Cost': 'Slightly lower',
        'Use Cases': 'Medium models (7-13B), balanced performance',
        'Examples': 'Llama 2 13B'
    },
    {
        'Technique': 'GQA-4',
        'Memory Efficiency': '8x reduction',
        'Quality': 'Very good (slight drop)',
        'Training Cost': 'Lower',
        'Use Cases': 'Large models (30-70B), production serving - RECOMMENDED',
        'Examples': 'Llama 2 70B, Mistral 7B'
    },
    {
        'Technique': 'GQA-2',
        'Memory Efficiency': '16x reduction',
        'Quality': 'Good (noticeable drop)',
        'Training Cost': 'Much lower',
        'Use Cases': 'Very large models (>70B), memory-critical applications',
        'Examples': 'Experimental large models'
    },
    {
        'Technique': 'MQA',
        'Memory Efficiency': '32x reduction',
        'Quality': 'Acceptable (clear drop)',
        'Training Cost': 'Lowest',
        'Use Cases': 'Code models, extreme memory constraints, fast inference priority',
        'Examples': 'PaLM, Falcon, StarCoder'
    },
    {
        'Technique': 'PagedAttention',
        'Memory Efficiency': '1.5x improvement',
        'Quality': 'No impact',
        'Training Cost': 'N/A (inference only)',
        'Use Cases': 'Production serving, high throughput, variable sequence lengths',
        'Examples': 'vLLM, TensorRT-LLM, TGI'
    },
    {
        'Technique': 'GQA + PagedAttention',
        'Memory Efficiency': '12x+ reduction',
        'Quality': 'Very good',
        'Training Cost': 'Lower + N/A',
        'Use Cases': 'Production standard for large models',
        'Examples': 'Llama 2 70B with vLLM'
    },
]

df = pd.DataFrame(decision_matrix)
print("\nKV Cache Optimization - Decision Matrix")
print("=" * 100)
print(df.to_string(index=False))

print("\n\nQuick Selection Guide:")
print("=" * 70)
print("""
1. For Training:
   - Model < 7B:    Use MHA (standard)
   - Model 7-30B:   Use GQA-8 or GQA-4
   - Model 30-70B:  Use GQA-4
   - Model > 70B:   Use GQA-2 or MQA
   - Code models:   Consider MQA (proven effective)

2. For Inference/Serving:
   - ALWAYS use PagedAttention (vLLM, TRT-LLM, or TGI)
   - Combine with GQA if model was trained with it
   - For legacy MHA models: Use PagedAttention alone

3. Migration Strategy:
   - Training new model: Choose GQA-4 or GQA-8
   - Have MHA model: Convert via uptraining (5-10% of original)
   - Production serving: Always deploy with PagedAttention

4. Quality vs Memory:
   - Quality critical: MHA or GQA-8
   - Balanced:         GQA-4 (recommended)
   - Memory critical:  GQA-2 or MQA
   - Fast inference:   MQA + PagedAttention
""")

## 10. Production Best Practices

In [ ]:
print("KV Cache Optimization - Production Best Practices")
print("=" * 70)

best_practices = """
1. ARCHITECTURE SELECTION:
   ✓ Default to GQA-4 for new large models (30B+)
   ✓ Use GQA-8 for medium models (7-13B) if quality is critical
   ✓ Consider MQA only for code models or extreme constraints
   ✓ Always benchmark on your specific tasks before committing

2. SERVING INFRASTRUCTURE:
   ✓ Use vLLM or TensorRT-LLM with PagedAttention (mandatory)
   ✓ Configure appropriate block size (16-32 tokens typical)
   ✓ Monitor memory fragmentation and utilization
   ✓ Enable continuous batching for maximum throughput

3. MEMORY MANAGEMENT:
   ✓ Calculate total KV cache budget: GPU_memory * 0.6-0.7
   ✓ Reserve ~30-40% for model weights and activations
   ✓ Set max_num_seqs based on average request length
   ✓ Implement request queueing and prioritization

4. MODEL CONVERSION:
   ✓ Converting MHA to GQA: Requires uptraining (5-10% of original)
   ✓ Mean-pool KV heads: k_new = mean(k_heads[group])
   ✓ Fine-tune on representative data mix
   ✓ Validate on comprehensive benchmarks

5. MONITORING:
   ✓ Track KV cache memory utilization
   ✓ Monitor cache hit rates (for prefix caching)
   ✓ Measure effective tokens per second
   ✓ Alert on OOM errors and cache thrashing

6. OPTIMIZATION CHECKLIST:
   ✓ Use fp16 or bf16 for KV cache (not fp32)
   ✓ Enable prefix caching for repeated prompts
   ✓ Implement KV cache quantization (int8) if needed
   ✓ Consider multi-tier caching (GPU + CPU memory)
   ✓ Batch requests with similar context lengths

7. COMMON PITFALLS:
   ✗ Using contiguous allocation (huge waste)
   ✗ Not monitoring memory fragmentation
   ✗ Over-allocating max sequence length
   ✗ Ignoring quality degradation with aggressive compression
   ✗ Not using PagedAttention in production

8. BENCHMARKING:
   ✓ Measure throughput (tokens/sec) at various batch sizes
   ✓ Test with realistic sequence length distributions
   ✓ Compare quality on task-specific metrics
   ✓ Profile memory usage under peak load
   ✓ Test cache efficiency with real traffic patterns
"""

print(best_practices)

# Example configuration template
print("\n" + "=" * 70)
print("Example vLLM Configuration (Llama 2 70B with GQA-4):")
print("=" * 70)

config = """
from vllm import LLM, SamplingParams

# Initialize with optimized settings
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    tensor_parallel_size=4,      # 4x A100 80GB
    dtype="bfloat16",             # Use bf16 for training stability
    max_model_len=4096,           # Maximum context length
    gpu_memory_utilization=0.90,  # Use 90% of GPU memory
    block_size=16,                # PagedAttention block size
    max_num_batched_tokens=8192,  # Continuous batching limit
    max_num_seqs=256,             # Max concurrent sequences
    enable_prefix_caching=True,   # Cache common prefixes
)

# With GQA-4, this configuration achieves:
# - 8x KV cache reduction vs MHA
# - 1.5x additional saving from PagedAttention
# - Total: ~12x memory efficiency improvement
# - Throughput: 2000+ tokens/sec with batch size 256
"""

print(config)

## Summary

### Key Takeaways

1. **KV Cache is the Bottleneck**:
   - For large models, KV cache dominates memory usage
   - Limits batch size and throughput significantly
   - Critical optimization target for production serving

2. **Attention Mechanism Evolution**:
   - **MHA**: Standard but memory-intensive
   - **MQA**: Maximum efficiency (32x) but quality tradeoff
   - **GQA**: Sweet spot - GQA-4 provides 8x reduction with minimal quality loss

3. **PagedAttention Revolution**:
   - Non-contiguous memory allocation like OS virtual memory
   - Near-zero memory waste
   - Enables efficient memory sharing
   - 1.5x additional improvement on top of GQA

4. **Production Standard**:
   - Train with GQA-4 for large models
   - Serve with vLLM/TensorRT-LLM (PagedAttention)
   - Combined: 12x memory reduction vs baseline MHA
   - 2-4x throughput improvement in practice

5. **Selection Guide**:
   - Small models: MHA
   - Medium models: GQA-8
   - Large models: GQA-4 (recommended)
   - Very large/code models: GQA-2 or MQA
   - Serving: Always use PagedAttention

### Historical Impact

- **2019**: MQA introduced but not widely adopted
- **2022**: Memory wall hit with 70B+ models
- **2023**: GQA + PagedAttention enable practical large model serving
- **2023-2024**: Became standard for all major models (Llama 2, Mistral, etc.)

### Next Steps

- **Notebook 56**: Speculative Decoding (2-3x speedup)
- **Notebook 57**: vLLM Deep Dive (production serving)
- **Notebook 58**: TensorRT-LLM (NVIDIA optimization)
- **Notebook 59**: Continuous Batching (throughput optimization)